# Dimension 2.3 - 20-Year Policy Scenario Simulation
System dynamics model, Monte Carlo uncertainty, 4 policy scenarios.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hdi.models.simulation import (
    SCENARIOS,
    simulate_scenario,
    monte_carlo_scenarios,
)
from hdi.visualization.charts import scenario_comparison

plt.rcParams["figure.dpi"] = 120
print("Imports OK")
print(f"Available scenarios: {list(SCENARIOS.keys())}")

## 1. Scenario Definitions

In [ ]:
# Display the 4 policy scenarios
for name, params in SCENARIOS.items():
    print(f"\n{'='*60}")
    print(f"Scenario: {name}")
    print(f"{'='*60}")
    for k, v in params.items():
        print(f"  {k:30s}: {v}")

## 2. Single Country Simulation (China)

In [ ]:
# Initial conditions for China (approximate 2023 values)
initial_conditions = {
    "pm25": 35.0,           # ug/m3
    "water_access": 0.85,   # fraction with safe water
    "sanitation": 0.78,     # fraction with improved sanitation
    "gdp_pc": 12_500,       # USD PPP
    "health_spend_pct": 5.4,# % of GDP
    "resp_daly_rate": 1200, # per 100k
    "diarrheal_rate": 180,  # per 100k
}

horizon = 20  # years

# Run all 4 scenarios
sim_results = {}
for scenario_name in SCENARIOS:
    sim_results[scenario_name] = simulate_scenario(
        scenario_name=scenario_name,
        initial_conditions=initial_conditions,
        horizon=horizon,
    )

# Plot key trajectories
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = ["pm25", "resp_daly_rate", "water_access", "diarrheal_rate"]
titles = ["PM2.5 (ug/m3)", "Respiratory DALY rate", "Safe water access", "Diarrheal rate"]

for ax, metric, title in zip(axes.flat, metrics, titles):
    for scenario_name, result in sim_results.items():
        years = result["years"]
        ax.plot(years, result[metric], label=scenario_name, lw=2)
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("China: 20-Year Policy Scenario Projections", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Monte Carlo Uncertainty

In [ ]:
# Run Monte Carlo for Scenario C (Accelerated Green Transition)
mc_result = monte_carlo_scenarios(
    scenario_name="C_accelerated_green",
    initial_conditions=initial_conditions,
    horizon=horizon,
    n_simulations=1000,
    seed=42,
)

# Extract percentiles for respiratory DALY
years = mc_result["years"]
median = mc_result["resp_daly_rate"]["p50"]
p05 = mc_result["resp_daly_rate"]["p05"]
p25 = mc_result["resp_daly_rate"]["p25"]
p75 = mc_result["resp_daly_rate"]["p75"]
p95 = mc_result["resp_daly_rate"]["p95"]

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(years, p05, p95, alpha=0.15, color="C0", label="90% CI")
ax.fill_between(years, p25, p75, alpha=0.30, color="C0", label="50% CI")
ax.plot(years, median, "C0-", lw=2, label="Median")

# Overlay BAU deterministic
ax.plot(years, sim_results["A_business_as_usual"]["resp_daly_rate"],
        "k--", lw=1.5, alpha=0.6, label="BAU (deterministic)")

ax.set_xlabel("Year")
ax.set_ylabel("Respiratory DALY rate (per 100k)")
ax.set_title("Scenario C: Monte Carlo Uncertainty (n=1000)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Year {years[-1]} median: {median[-1]:.0f} (90% CI: {p05[-1]:.0f}-{p95[-1]:.0f})")

## 4. Multi-Country Comparison

In [ ]:
# Define initial conditions for 4 countries
country_params = {
    "China": {
        "pm25": 35.0, "water_access": 0.85, "sanitation": 0.78,
        "gdp_pc": 12_500, "health_spend_pct": 5.4,
        "resp_daly_rate": 1200, "diarrheal_rate": 180,
    },
    "India": {
        "pm25": 55.0, "water_access": 0.72, "sanitation": 0.60,
        "gdp_pc": 7_200, "health_spend_pct": 3.5,
        "resp_daly_rate": 2100, "diarrheal_rate": 850,
    },
    "USA": {
        "pm25": 8.5, "water_access": 0.99, "sanitation": 0.99,
        "gdp_pc": 63_000, "health_spend_pct": 17.0,
        "resp_daly_rate": 450, "diarrheal_rate": 15,
    },
    "Nigeria": {
        "pm25": 48.0, "water_access": 0.42, "sanitation": 0.35,
        "gdp_pc": 5_100, "health_spend_pct": 3.0,
        "resp_daly_rate": 2800, "diarrheal_rate": 2200,
    },
}

# Simulate all countries x all scenarios
multi_results = {}
for country, ic in country_params.items():
    multi_results[country] = {}
    for scenario_name in SCENARIOS:
        multi_results[country][scenario_name] = simulate_scenario(
            scenario_name=scenario_name,
            initial_conditions=ic,
            horizon=horizon,
        )

# Use visualization helper
fig = scenario_comparison(
    multi_results,
    metric="resp_daly_rate",
    title="Respiratory DALY Projections by Country & Scenario",
)
plt.show()

# Summary table: endpoint values under Scenario C
summary_rows = []
for country in country_params:
    baseline = country_params[country]["resp_daly_rate"]
    endpoint = multi_results[country]["C_accelerated_green"]["resp_daly_rate"][-1]
    reduction = (baseline - endpoint) / baseline * 100
    summary_rows.append({
        "Country": country,
        "Baseline DALY rate": baseline,
        "Endpoint DALY rate (Sc. C)": round(endpoint, 1),
        "Reduction %": round(reduction, 1),
    })

print(pd.DataFrame(summary_rows).to_string(index=False))

## 5. Sensitivity Analysis (Tornado Diagram)

In [ ]:
# One-at-a-time sensitivity analysis on key parameters
# Perturb each parameter +/-20% and measure impact on endpoint respiratory DALY

base_ic = country_params["China"].copy()
baseline_result = simulate_scenario(
    "C_accelerated_green", base_ic, horizon=horizon
)
baseline_endpoint = baseline_result["resp_daly_rate"][-1]

perturb_params = ["pm25", "water_access", "sanitation", "gdp_pc",
                  "health_spend_pct", "resp_daly_rate", "diarrheal_rate"]
perturbation = 0.20  # +/- 20%

sensitivity = []
for param in perturb_params:
    for direction, factor in [("low", 1 - perturbation), ("high", 1 + perturbation)]:
        ic_mod = base_ic.copy()
        ic_mod[param] = base_ic[param] * factor
        # Clamp fractions to [0, 1]
        if param in ["water_access", "sanitation"]:
            ic_mod[param] = min(1.0, max(0.0, ic_mod[param]))

        res = simulate_scenario("C_accelerated_green", ic_mod, horizon=horizon)
        endpoint = res["resp_daly_rate"][-1]
        sensitivity.append({
            "param": param,
            "direction": direction,
            "endpoint": endpoint,
            "delta": endpoint - baseline_endpoint,
        })

sens_df = pd.DataFrame(sensitivity)

# Build tornado data
tornado = []
for param in perturb_params:
    low_val = sens_df[(sens_df["param"] == param) & (sens_df["direction"] == "low")]["endpoint"].values[0]
    high_val = sens_df[(sens_df["param"] == param) & (sens_df["direction"] == "high")]["endpoint"].values[0]
    tornado.append({
        "param": param,
        "low": low_val,
        "high": high_val,
        "swing": abs(high_val - low_val),
    })

tornado_df = pd.DataFrame(tornado).sort_values("swing", ascending=True)

# Plot tornado diagram
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(tornado_df))

for i, row in enumerate(tornado_df.itertuples()):
    left = min(row.low, row.high)
    width = abs(row.high - row.low)
    color = "steelblue" if row.high > row.low else "coral"
    ax.barh(i, row.high - baseline_endpoint, left=baseline_endpoint,
            height=0.6, color="coral", alpha=0.8)
    ax.barh(i, row.low - baseline_endpoint, left=baseline_endpoint,
            height=0.6, color="steelblue", alpha=0.8)

ax.axvline(baseline_endpoint, color="k", ls="-", lw=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(tornado_df["param"])
ax.set_xlabel("Endpoint Respiratory DALY rate (per 100k)")
ax.set_title("Tornado Diagram: Parameter Sensitivity (±20% perturbation)")

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="coral", alpha=0.8, label="+20% perturbation"),
    Patch(color="steelblue", alpha=0.8, label="-20% perturbation"),
], loc="lower right")

plt.tight_layout()
plt.show()

print("\nParameter sensitivity ranking (largest swing first):")
print(tornado_df[["param", "swing"]].sort_values("swing", ascending=False).to_string(index=False))